In [1]:
!pip -q install -U protobuf sentencepiece tokenizers "huggingface_hub>=0.24.0" "diffusers>=0.30.0" "transformers>=4.40.0" accelerate safetensors pillow tqdm

In [2]:
!pip -q install git+https://github.com/huggingface/diffusers.git
!pip -q install --upgrade torch torchvision transformers accelerate bitsandbytes

In [3]:

# Option A: interactive (prompts in notebook)
# Fix dependency mismatch


from huggingface_hub import login
login()  # paste your HF token when prompted

# Option B: non-interactive (set token explicitly)
# import os
# os.environ["HF_TOKEN"] = "hf_..."   # <-- put your token here (or load from env/secret store)
# login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

# Option C: if you already did `huggingface-cli login` in terminal, you can skip login().


In [1]:
import os, json, time, random, glob, math
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple
from PIL import Image
from tqdm.auto import tqdm

import torch

# Speed knobs (safe defaults for high-VRAM GPU boxes)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
CIFAR10_CLASSES = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

@dataclass
class GenConfig:
    jsonl_paths: List[str]
    out_root: str = "./cifar10_synth"
    samples_per_class: int = 2000
    seed: int = 123

    # Generate high-res, then downsample to CIFAR-10 32x32
    gen_width: int = 512
    gen_height: int = 512
    cifar_size: int = 32

    # Inference
    steps: int = 30
    guidance: float = 7.5

    # Performance
    batch_size: int = 100           # increase on high VRAM (e.g., 8, 12, 16)
    num_gpus: Optional[int] = None # None = use all visible GPUs

CFG = GenConfig(
    jsonl_paths=["airplane.jsonl"],   # <-- change me
    out_root="./cifar10_synth_flux",
    samples_per_class=1000,
    seed=123,
    gen_width=512, gen_height=512,
    cifar_size=32,
    steps=30, guidance=7.5,
    batch_size=100,                 # <-- tune for your VRAM
    num_gpus=None,                # use all GPUs by default
)

os.makedirs(CFG.out_root, exist_ok=True)

In [3]:
def read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Bad JSON on {path}:{ln}: {e}")

def load_rows(paths: List[str]) -> List[Dict[str, Any]]:
    rows = []
    for p in paths:
        rows.extend(list(read_jsonl(p)))
    return rows

def normalize_label(s: str) -> str:
    return (s or "").strip().lower()

def group_by_class(rows: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    out = {c: [] for c in CIFAR10_CLASSES}
    other: Dict[str, int] = {}
    for r in rows:
        cls = normalize_label(r.get("class_label"))
        if cls in out:
            out[cls].append(r)
        else:
            other[cls] = other.get(cls, 0) + 1
    if other:
        print("Warning: non-CIFAR class_label(s) found (ignored):")
        for k, v in other.items():
            print(f"  - {k!r}: {v}")
    return out

def stable_sample(items: List[Dict[str, Any]], k: int, rng: random.Random) -> List[Dict[str, Any]]:
    if len(items) <= k:
        return list(items)
    idx = list(range(len(items)))
    rng.shuffle(idx)
    return [items[i] for i in idx[:k]]

rows = load_rows(CFG.jsonl_paths)
by_cls = group_by_class(rows)

rng = random.Random(CFG.seed)
sampled: Dict[str, List[Dict[str, Any]]] = {}
for c in CIFAR10_CLASSES:
    sampled[c] = stable_sample(by_cls[c], CFG.samples_per_class, rng)
    print(f"{c:>10}: sampled {len(sampled[c])} / available {len(by_cls[c])}")

  airplane: sampled 1000 / available 1000
automobile: sampled 0 / available 0
      bird: sampled 0 / available 0
       cat: sampled 0 / available 0
      deer: sampled 0 / available 0
       dog: sampled 0 / available 0
      frog: sampled 0 / available 0
     horse: sampled 0 / available 0
      ship: sampled 0 / available 0
     truck: sampled 0 / available 0


In [4]:
def safe_filename(s: str) -> str:
    s = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in (s or ""))
    s = s.strip("_")
    return (s[:120] or "item")

def resize_to_cifar(img: Image.Image, size: int) -> Image.Image:
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img.resize((size, size), resample=Image.Resampling.LANCZOS)

def build_tasks(sampled_by_class: Dict[str, List[Dict[str, Any]]], backend_name: str, cfg: GenConfig):
    tasks = []
    for cls, items in sampled_by_class.items():
        for r in items:
            prompt = r.get("prompt") or ""
            neg = r.get("negative_prompt") or None
            pid = r.get("id", None)

            base = f"{cfg.seed}|{backend_name}|{cls}|{pid}|{prompt}"
            seed = abs(hash(base)) % (2**31 - 1)

            stem = safe_filename(str(pid)) if pid is not None else safe_filename(prompt[:80])

            # Modified paths here to put class folders directly under cifar_hires and cifar32
            hires_dir = os.path.join(cfg.out_root, "cifar_hires", cls)
            cifar_dir = os.path.join(cfg.out_root, "cifar32", cls)

            hires_path = os.path.join(hires_dir, f"{stem}_{seed}.png")
            cifar_path = os.path.join(cifar_dir, f"{stem}_{seed}.png")

            tasks.append({
                "class_label": cls,
                "prompt_id": pid,
                "prompt": prompt,
                "negative_prompt": neg,
                "seed": int(seed),
                "hires_path": hires_path,
                "cifar_path": cifar_path,
            })
    return tasks

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

In [5]:
import torch
from diffusers import Flux2Pipeline, AutoModel
from transformers import Mistral3ForConditionalGeneration, BitsAndBytesConfig

# Configuration
BACKEND_KIND = "flux2"
BACKEND_NAME = "flux2_dev"
# The source for the 4-bit quantized weights
QUANT_REPO_ID = "diffusers/FLUX.2-dev-bnb-4bit" 
# The original model ID for other components if needed
MODEL_ID = "black-forest-labs/FLUX.2-dev"
device = "cuda:0"
DTYPE = torch.bfloat16

assert torch.cuda.is_available(), "CUDA GPU not available."

### 1. Load the 4-bit Text Encoder
# We load this separately to ensure it uses the quantized weights
text_encoder = Mistral3ForConditionalGeneration.from_pretrained(
    QUANT_REPO_ID, 
    subfolder="text_encoder", 
    torch_dtype=DTYPE, 
    device_map="auto"
)

### 2. Load the 4-bit Transformer (DiT)
transformer = AutoModel.from_pretrained(
    QUANT_REPO_ID, 
    subfolder="transformer", 
    torch_dtype=DTYPE, 
    device_map="auto"
)

### 3. Initialize the Pipeline
# We pass the pre-loaded 4-bit components into the pipeline
pipe = Flux2Pipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    transformer=transformer,
    torch_dtype=DTYPE,
    use_safetensors=True,
)

# Use CPU offloading to further reduce VRAM usage if necessary
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)

# Optional VAE tweaks for memory efficiency
try:
    if hasattr(pipe, "vae"):
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()
except Exception as e:
    print(f"VAE Tweak failed: {e}")

print("Loaded 4-bit components from:", QUANT_REPO_ID)
print("Base Model ID:", MODEL_ID, "| dtype:", DTYPE, "| device:", device)



config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model_index.json:   0%|          | 0.00/431 [00:00<?, ?B/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded 4-bit components from: diffusers/FLUX.2-dev-bnb-4bit
Base Model ID: black-forest-labs/FLUX.2-dev | dtype: torch.bfloat16 | device: cuda:0


In [ ]:
import os, json, time
from tqdm.auto import tqdm
import torch

tasks = build_tasks(sampled, BACKEND_NAME, CFG)
meta_path = os.path.join(CFG.out_root, f"metadata_{BACKEND_NAME}.jsonl")
os.makedirs(CFG.out_root, exist_ok=True)
# CFG.batch_size = 8
def _pipe_call_flux2(prompts, negs, gens):
    """
    FLUX.2 pipeline call (local text encoder).
    Many FLUX pipelines do NOT use negative_prompt; we ignore it by default.
    If your Flux2Pipeline supports negative_prompt, you can try passing it.
    """
    kwargs = dict(
        prompt=prompts,
        num_inference_steps=CFG.steps,
        guidance_scale=CFG.guidance,
        generator=gens,
        width=CFG.gen_width,
        height=CFG.gen_height,
    )
    return pipe(**kwargs)

print(f"Total tasks: {len(tasks)} | batch_size={CFG.batch_size}")
pbar = tqdm(total=len(tasks))

with open(meta_path, "a", encoding="utf-8") as mf:
    for batch0 in chunked(tasks, CFG.batch_size):
        # skip if both images already exist
        batch = [t for t in batch0 if not (os.path.exists(t["hires_path"]) and os.path.exists(t["cifar_path"]))]

        if not batch:
            pbar.update(len(batch0))
            continue

        for t in batch:
            os.makedirs(os.path.dirname(t["hires_path"]), exist_ok=True)
            os.makedirs(os.path.dirname(t["cifar_path"]), exist_ok=True)

        prompts = [t["prompt"] for t in batch]
        negs = [t["negative_prompt"] for t in batch]
        gens = [torch.Generator(device=device).manual_seed(int(t["seed"])) for t in batch]

        t0 = time.time()
        try:
            with torch.inference_mode():
                out = _pipe_call_flux2(prompts, negs, gens)

            images = out.images

            for tsk, img in zip(batch, images):
                # Save high-res
                img_rgb = img.convert("RGB") if img.mode != "RGB" else img
                img_rgb.save(tsk["hires_path"], format="PNG", optimize=True)

                # Save CIFAR-10 sized 32x32
                img32 = resize_to_cifar(img_rgb, CFG.cifar_size)
                img32.save(tsk["cifar_path"], format="PNG", optimize=True)

                rec = {
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "backend": BACKEND_NAME,
                    "kind": BACKEND_KIND,
                    "model_id": MODEL_ID,
                    "class_label": tsk["class_label"],
                    "hires_path": tsk["hires_path"],
                    "cifar32_path": tsk["cifar_path"],
                    "seed": tsk["seed"],
                    "steps": CFG.steps,
                    "guidance": CFG.guidance,
                    "gen_width": CFG.gen_width,
                    "gen_height": CFG.gen_height,
                    "cifar_size": CFG.cifar_size,
                    "prompt_id": tsk["prompt_id"],
                    "prompt": tsk["prompt"],
                    "negative_prompt": tsk["negative_prompt"],
                    "seconds_batch": round(time.time() - t0, 3),
                }
                mf.write(json.dumps(rec, ensure_ascii=False) + "\n")
            mf.flush()

        except Exception as e:
            for tsk in batch:
                mf.write(json.dumps({
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "backend": BACKEND_NAME,
                    "kind": BACKEND_KIND,
                    "model_id": MODEL_ID,
                    "class_label": tsk["class_label"],
                    "prompt_id": tsk["prompt_id"],
                    "prompt": tsk["prompt"],
                    "error": repr(e),
                }, ensure_ascii=False) + "\n")
            mf.flush()

        pbar.update(len(batch0))

pbar.close()
print(f"Done. Images in: {os.path.join(CFG.out_root, BACKEND_NAME)}")
print(f"Metadata appended to: {meta_path}")


Total tasks: 1000 | batch_size=100


  0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:


# 3. Clear the PyTorch CUDA cache (this releases memory back to the GPU)
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi


Mon Jan 19 20:36:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:00:08.0 Off |                    0 |
| N/A   38C    P0             48W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
torch.cuda.empty_cache()

In [ ]:
!nvidia-smi